In [3]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [5]:
import os
import shutil

import numpy as np
from torchvision import datasets, transforms

# Set the directory where you want to save the files
save_dir = "data"
os.makedirs(save_dir, exist_ok=True)

# Download and load the MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])
mnist_train = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
mnist_test = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# Convert to numpy arrays and normalize
X_train = mnist_train.data.numpy().reshape(-1, 28 * 28).astype(np.float32) / 255.0
y_train = mnist_train.targets.numpy().astype(np.int32)
X_test = mnist_test.data.numpy().reshape(-1, 28 * 28).astype(np.float32) / 255.0
y_test = mnist_test.targets.numpy().astype(np.int32)

# Save the data as raw binary files
X_train.tofile(os.path.join(save_dir, "X_train.bin"))
y_train.tofile(os.path.join(save_dir, "y_train.bin"))
X_test.tofile(os.path.join(save_dir, "X_test.bin"))
y_test.tofile(os.path.join(save_dir, "y_test.bin"))

# Save metadata
with open(os.path.join(save_dir, "metadata.txt"), "w") as f:
    f.write(f"Training samples: {X_train.shape[0]}\n")
    f.write(f"Test samples: {X_test.shape[0]}\n")
    f.write(f"Input dimensions: {X_train.shape[1]}\n")
    f.write(f"Number of classes: {len(np.unique(y_train))}\n")

# Remove mnist_data directory if it exists
if os.path.exists("mnist_data"):
    shutil.rmtree("mnist_data")
    print("Removed mnist_data directory")

# Remove MNIST raw data directory after processing
if os.path.exists(os.path.join(save_dir, "MNIST")):
    shutil.rmtree(os.path.join(save_dir, "MNIST"))
    print("Removed MNIST raw data directory")

print("MNIST dataset has been downloaded and saved in binary format in data/ directory.")

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:02<00:00, 3.67MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 115kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:01<00:00, 1.12MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 8.82MB/s]


Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

Removed MNIST raw data directory
MNIST dataset has been downloaded and saved in binary format in data/ directory.


In [9]:
TRAIN_SIZE = 10000
epochs = 10
learning_rate = 1e-2
batch_size = 32

torch.set_float32_matmul_precision('high')

#Load binary data
X_train_np = np.fromfile("data/X_train.bin", dtype=np.float32).reshape(60000, 784)
y_train_np = np.fromfile("data/y_train.bin", dtype=np.int32)
X_test_np = np.fromfile("data/X_test.bin", dtype=np.float32).reshape(10000, 784)
y_test_np = np.fromfile("data/y_test.bin", dtype=np.int32)

# Apply MNIST normalization (mean=0.1307, std=0.3081)
mean, std = 0.1307, 0.3081
X_train_np = (X_train_np - mean) / std
X_test_np = (X_test_np - mean) / std

# Convert to PyTorch tensors and move to GPU immediately
train_data = torch.from_numpy(X_train_np[:TRAIN_SIZE].reshape(-1, 1, 28, 28)).to("cuda")
train_labels = torch.from_numpy(y_train_np[:TRAIN_SIZE]).long().to("cuda")
test_data = torch.from_numpy(X_test_np.reshape(-1, 1, 28, 28)).to("cuda")
test_labels = torch.from_numpy(y_test_np).long().to("cuda")

iters_per_epoch = TRAIN_SIZE // batch_size

In [11]:
class MLP(nn.Module):
    def __init__(self, in_features, hidden_features, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_features, num_classes)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)  # Flatten the input
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
    
# Use random seed for natural variance (no fixed seed)
model = MLP(in_features=784, hidden_features=1024, num_classes=10).to("cuda")

In [13]:
# Apply custom He initialization to match NumPy/C implementations
with torch.no_grad():   # Disable gradient tracking for initialization
    # Layer 1: He initialization for ReLU
    fan_in = model.fc1.weight.size(1)  # input features
    scale = (2.0 / fan_in) ** 0.5 # He initialization scale factor
    model.fc1.weight.uniform_(-scale, scale) #fills weight with values from uniform distribution in range [-scale, scale]
    model.fc1.bias.zero_()
    
    # Layer 2: He initialization for ReLU  
    fan_in = model.fc2.weight.size(1)  # hidden features
    scale = (2.0 / fan_in) ** 0.5 # He initialization scale factor so RelU variance is preserved
    model.fc2.weight.uniform_(-scale, scale)
    model.fc2.bias.zero_()

# model = torch.compile(model)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# Training the model with detailed timing
def train_timed(model, criterion, optimizer, epoch, timing_stats, epoch_losses):
    model.train()
    epoch_loss = 0.0
    
    for i in range(iters_per_epoch):
        # Data loading timing (no GPU transfer needed - already on GPU)
        data_start = time.time()
        data = train_data[i * batch_size : (i + 1) * batch_size]
        target = train_labels[i * batch_size : (i + 1) * batch_size]
        data_end = time.time()
        timing_stats['data_loading'] += data_end - data_start   #time taken to slice the batch from preloaded GPU tensors
        
        optimizer.zero_grad()   #claers previous gradients for the next iteration
        
        # Forward pass timing
        forward_start = time.time()
        outputs = model(data)
        forward_end = time.time()
        timing_stats['forward'] += forward_end - forward_start
        
        # Loss computation timing
        loss_start = time.time()
        loss = criterion(outputs, target)
        epoch_loss += loss.item()
        loss_end = time.time()
        timing_stats['loss_computation'] += loss_end - loss_start
        
        # Backward pass timing
        backward_start = time.time()
        loss.backward()
        backward_end = time.time()
        timing_stats['backward'] += backward_end - backward_start
        
        # Weight updates timing
        update_start = time.time()
        optimizer.step()
        optimizer.zero_grad()
        update_end = time.time()
        timing_stats['weight_updates'] += update_end - update_start
    
    # Store average loss for this epoch
    epoch_losses.append(epoch_loss / iters_per_epoch)

In [15]:
# Evaluation function to report average batch accuracy using the loaded test data
def evaluate(model, test_data, test_labels):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    total_batch_accuracy = torch.tensor(0.0, device=device)
    num_batches = 0

    with torch.no_grad():
        for i in range(len(test_data) // batch_size):
            data = test_data[i * batch_size : (i + 1) * batch_size]
            target = test_labels[i * batch_size : (i + 1) * batch_size]
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            correct_batch = (predicted == target).sum().item()
            total_batch = target.size(0)
            if total_batch != 0:  # Check to avoid division by zero
                batch_accuracy = correct_batch / total_batch
                total_batch_accuracy += batch_accuracy
                num_batches += 1

    avg_batch_accuracy = total_batch_accuracy / num_batches
    print(f"Average Batch Accuracy: {avg_batch_accuracy * 100:.2f}%")


In [16]:
# Main
if __name__ == "__main__":
    # Initialize timing stats and loss tracking
    timing_stats = {
        'data_loading': 0.0,
        'forward': 0.0,
        'loss_computation': 0.0,
        'backward': 0.0,
        'weight_updates': 0.0,
        'total_time': 0.0
    }
    epoch_losses = []
    
    # Start total timing
    total_start = time.time()
    
    for epoch in range(epochs):
        train_timed(model, criterion, optimizer, epoch, timing_stats, epoch_losses)
        print(f"Epoch {epoch} loss: {epoch_losses[epoch]:.4f}")

    # End total timing
    total_end = time.time()
    timing_stats['total_time'] = total_end - total_start
    
    # Print detailed timing breakdown
    print("\n=== PYTORCH CUDA IMPLEMENTATION TIMING BREAKDOWN ===")
    print(f"Total training time: {timing_stats['total_time']:.1f} seconds\n")
    
    print("Detailed Breakdown:")
    print(f"  Data loading:     {timing_stats['data_loading']:6.3f}s ({100.0 * timing_stats['data_loading'] / timing_stats['total_time']:5.1f}%)")
    print(f"  Forward pass:     {timing_stats['forward']:6.3f}s ({100.0 * timing_stats['forward'] / timing_stats['total_time']:5.1f}%)")
    print(f"  Loss computation: {timing_stats['loss_computation']:6.3f}s ({100.0 * timing_stats['loss_computation'] / timing_stats['total_time']:5.1f}%)")
    print(f"  Backward pass:    {timing_stats['backward']:6.3f}s ({100.0 * timing_stats['backward'] / timing_stats['total_time']:5.1f}%)")
    print(f"  Weight updates:   {timing_stats['weight_updates']:6.3f}s ({100.0 * timing_stats['weight_updates'] / timing_stats['total_time']:5.1f}%)")

    print("Finished Training")

Epoch 0 loss: 0.6916
Epoch 1 loss: 0.3449
Epoch 2 loss: 0.2840
Epoch 3 loss: 0.2485
Epoch 4 loss: 0.2225
Epoch 5 loss: 0.2017
Epoch 6 loss: 0.1841
Epoch 7 loss: 0.1689
Epoch 8 loss: 0.1555
Epoch 9 loss: 0.1435

=== PYTORCH CUDA IMPLEMENTATION TIMING BREAKDOWN ===
Total training time: 4.8 seconds

Detailed Breakdown:
  Data loading:      0.102s (  2.1%)
  Forward pass:      0.972s ( 20.4%)
  Loss computation:  0.715s ( 15.0%)
  Backward pass:     2.222s ( 46.7%)
  Weight updates:    0.642s ( 13.5%)
Finished Training
